# Notebook 07: Introduction to AI Models

## Before You Start
> **If anything behaves unexpectedly, restart the kernel first: Kernel menu → Restart Kernel and Clear All Outputs. Then run the cells from the top.**

## ADAS Connection
In the last three notebooks your robot learned to **see** -- detecting colors and following targets using computer vision. But computer vision alone has limits. It can tell you *what* is in the frame but not always *what to do about it*.

Modern autonomous vehicles use **AI models** as a decision-making layer on top of their sensors. The sensors feed data in, the model reasons about it, and a decision comes out. Tesla calls this their **neural network stack**. Waymo calls it their **driver**.

In this notebook you will meet the AI model that will become your robot's brain -- **Phi-3 Mini**, a small but capable language model running on a dedicated AI server. You will learn how to talk to it, how to give it context, and how to get useful decisions out of it.

---

 ## How It Works

Your robot connects to an AI server over the school network. The server runs two pieces
of software working together:

**Ollama** is an open source tool that makes it possible to run AI models locally -- on
a regular laptop or server. Instead of sending your data to OpenAI or Google, the model
runs right here in the room on the instructor's laptop. Your robot talks to it over the
school WiFi network. This matters for autonomous vehicles because real self-driving cars
try to minimize dependence on distant cloud servers -- keeping inference close to the
vehicle reduces latency and means decisions come back faster.

**Phi-3 Mini** is the AI model running inside Ollama. It was developed by Microsoft and is designed to be small enough to run on modest hardware while still being capable of complex reasoning. "Mini" refers to its size -- it has 3.8 billion paramters, which sounds large but is tiny compared to models like GPT-4 which have over a trillion. The tradeoff is speed and efficiency over raw capability, which is exactly what you want in an embedded system.

---

## The Code
Run this cell to connect to the AI server. But first let's look at the ai_driver.py helper code.

In [1]:
import sys
import requests
import json
import time
sys.path.insert(0, '/home/pi/lab')
import ai_driver

# Test connection to AI server
print('Connecting to AI server...')
try:
    response = requests.get(f'http://{ai_driver.OLLAMA_HOST}:{ai_driver.OLLAMA_PORT}', timeout=5)
    print(f'Connected! Server is running at {ai_driver.OLLAMA_HOST}:{ai_driver.OLLAMA_PORT}')
    print(f'Model: {ai_driver.MODEL}')
    # Warm up the model -- first call loads it into memory
    print('Warming up AI model...')
    ai_driver.ask("hello")
    print('AI model ready!')
except Exception as e:
    print(f'ERROR: Could not reach AI server -- {e}')
    print(f'Make sure the MacBook is on the same network and Ollama is running.')

Connecting to AI server...
Connected! Server is running at 192.168.4.44:11434
Model: phi3:mini
Warming up AI model...
AI model ready!


---

## YOUR TURN -- Tweak Zone 1: Ask the Model Anything

Type any question in the PROMPT variable and run the cell. This is the same interface your robot will use to make driving decisions.

Try a few questions related to what we have been building:
- "What is ADAS?"
- "What does a self-driving car do when it sees a red light?"
- "What is computer vision?"

> **Think like an engineer:** Notice the response time. This is the latency your robot will experience when asking the model for a driving decision. How would latency affect a real self-driving car at highway speed?

In [2]:
# ═══════════════════════════════════════
#   TWEAK THIS VALUE -- ask the model anything
PROMPT = "What is ADAS in one sentence?"
# ═══════════════════════════════════════

print(f'Asking: {PROMPT}')
print('─' * 50)
start = time.time()
response = ai_driver.ask(PROMPT)
elapsed = time.time() - start
print(response)
print('─' * 50)
print(f'Response time: {elapsed:.2f} seconds')

Asking: What is ADAS in one sentence?
──────────────────────────────────────────────────
ADAS, or Advanced Driver-Assistance Systems, are technologies developed to enhance car safety and improve the driving experience by offering features like adaptive cruise control, lane keeping assistance, and collision avoidance.
──────────────────────────────────────────────────
Response time: 1.66 seconds


---

## YOUR TURN -- Tweak Zone 2: Give the Model Context

A raw question gives a general answer. But when you give the model **context** -- information about the situation -- it gives a much more specific and useful answer.

This is called **prompt engineering** -- designing your prompt to get the best response. It is one of the most important skills in working with AI models.

Change the CONTEXT and QUESTION variables below. Notice how the same question gets a different answer when you change the context.

> **Think like an engineer:** In a real ADAS system, what context would you give the model? What does it need to know to make a good driving decision?

In [3]:
# ═══════════════════════════════════════
#   TWEAK THESE VALUES
CONTEXT  = "You are the AI brain of a small robot car in a classroom. Give a brief, practical answer in 2-3 sentences."
QUESTION = "The camera sees a red object on the left side of the frame. What should the robot do?"
# ═══════════════════════════════════════

prompt = f"{CONTEXT}\n\n{QUESTION}"

print(f'Context: {CONTEXT}')
print(f'Question: {QUESTION}')
print('─' * 50)
start = time.time()
response = ai_driver.ask(prompt)
elapsed = time.time() - start
print(response)
print('─' * 50)
print(f'Response time: {elapsed:.2f} seconds')

Context: You are the AI brain of a small robot car in a classroom. Give a brief, practical answer in 2-3 sentences.
Question: The camera sees a red object on the left side of the frame. What should the robot do?
──────────────────────────────────────────────────
Based on visual cues and assuming it is within reach without disrupting activities or breaking any rules: The robot could gently pick up the red object using its built-in tools, if applicable, for a classroom demonstration or to add an element of interaction with students. Otherwise, maintain awareness and continue observing other objects that may be relevant during ongoing lessons.
──────────────────────────────────────────────────
Response time: 2.57 seconds


Notice that longer, vaguer prompts get slower and less useful responses.
Try shortening your CONTEXT and QUESTION to get a faster, more direct answer.
This is the core skill of prompt engineering.

---

## YOUR TURN -- Tweak Zone 3: Get a Driving Decision

Now let's ask the model for a structured driving decision -- exactly what the robot needs.

The key is telling the model to respond with **only one word**. Without this constraint the model gives long explanations, but the robot needs a single clear command.

Change the OBSERVATION variable to describe different situations and see what the model decides.

> **Think like an engineer:** What happens if the model doesn't follow the one-word instruction? How would you handle an unexpected response in your robot code?

In [4]:
# ═══════════════════════════════════════
# DEFAULT PROMPT: "You are the AI brain of a small robot car following a colored target.
#   Respond with ONLY one word: FORWARD, LEFT, RIGHT, or STOP.
#.  No explanation. No punctuation. Just the single word." 
# FRAME REFERENCE: 
#   X=0              X=320             X=640
#   |----LEFT----|---CENTER---|----RIGHT----|
#   TWEAK THIS VALUE -- describe what the robot sees
OBSERVATION = "No target detected. Frame is empty."

# try:
# "The target color is detected in the center of the frame at X position 310."
# "The target color is detected on the right side of the frame at X position 520."
# "No color detected. The frame is empty."
# "An obstacle is detected 20cm ahead."
# MORE OBSERVATIONS TO TRY:
# "Target color detected in the center at X=315 out of 640."
# "Target color detected on the right side at X=540 out of 640."
# "No target detected. Frame is empty."
# "Target detected but very small, radius=5px. Possibly far away."
# "Target detected but very large, radius=180px. Very close."
# "Obstacle detected 15cm ahead by ultrasonic sensor."
# "Obstacle detected 60cm ahead. Target color visible at X=200."
# "Target was detected last frame at X=400, now lost."
# ═══════════════════════════════════════

print(f'Observation: {OBSERVATION}')
print('─' * 50)
start = time.time()
decision = ai_driver.decide(OBSERVATION)
elapsed = time.time() - start
print(f'Decision: {decision}')
print('─' * 50)
print(f'Response time: {elapsed:.2f} seconds')
print()
if decision in ['FORWARD', 'LEFT', 'RIGHT', 'STOP']:
    print(f'Valid decision -- robot would: {decision}')
else:
    print(f'Unexpected response: "{decision}" -- robot would STOP for safety')

Observation: No target detected. Frame is empty.
──────────────────────────────────────────────────
Decision: STOP
──────────────────────────────────────────────────
Response time: 0.33 seconds

Valid decision -- robot would: STOP


---

## YOUR TURN -- Tweak Zone 4: Improve the Prompt

The `decide()` function in ai_driver.py has a built-in prompt. But you can write a better one.

Edit the SYSTEM_PROMPT below to give the model better instructions. Try to get more consistent and accurate decisions.

> **Team challenge:** Which team can write a prompt that gets the correct decision every time for all four observations below? Test each one.

In [5]:
# ═══════════════════════════════════════
#   TWEAK THE SYSTEM PROMPT
SYSTEM_PROMPT = """You are the AI brain of a small robot car.
Given an observation about what the robot's camera sees, respond with ONLY one word.
Your response must be exactly one of: FORWARD, LEFT, RIGHT, STOP
No explanation. No punctuation. Just the single word."""
# ═══════════════════════════════════════

test_observations = [
    "Target detected on the left side at X=100",
    "Target detected in the center at X=315",  
    "Target detected on the right side at X=540",
    "No target detected",
]

print('Testing prompt with 4 observations:')
print('─' * 50)
for obs in test_observations:
    prompt = f"{SYSTEM_PROMPT}\n\nObservation: {obs}"
    decision = ai_driver.ask(prompt).strip().upper()
    valid = decision in ['FORWARD', 'LEFT', 'RIGHT', 'STOP']
    status = 'PASS' if valid else 'FAIL'
    print(f'[{status}] {obs[:45]:<45} → {decision}')
print('─' * 50)

Testing prompt with 4 observations:
──────────────────────────────────────────────────
[PASS] Target detected on the left side at X=100     → LEFT
[PASS] Target detected in the center at X=315        → FORWARD
[PASS] Target detected on the right side at X=540    → RIGHT
[PASS] No target detected                            → STOP
──────────────────────────────────────────────────


---

## What Happened?

Think about these questions with your team:

1. Did the model always give a valid one-word response? What happened when it didn't?
2. How did changing the prompt affect the quality of the decisions?
3. Why is it important to validate the model's response before acting on it?
4. A real self-driving car cannot afford to get a bad response from its AI. What safety mechanisms would you build in?

---

## CHALLENGE -- Advanced Students

The current `decide()` function only returns one of four commands. Real ADAS systems have many more possible actions.

Extend the decision set to include: FORWARD_SLOW, FORWARD_FAST, LEFT_SHARP, LEFT_GENTLE, RIGHT_SHARP, RIGHT_GENTLE, STOP, REVERSE.

Write a new prompt that gets the model to use these extended commands correctly. Then write the Python code that maps each command to actual motor speeds.

In [ ]:
# YOUR CODE HERE
# ═══════════════════════════════════════
EXTENDED_COMMANDS = ['FORWARD_SLOW', 'FORWARD_FAST', 'LEFT_SHARP', 
                     'LEFT_GENTLE', 'RIGHT_SHARP', 'RIGHT_GENTLE', 
                     'STOP', 'REVERSE']
# ═══════════════════════════════════════

def extended_decide(observation):
    # Write your prompt here
    # Return one of the EXTENDED_COMMANDS
    pass

def execute_extended(command):
    # Map each command to motor speeds
    # Use motors.forward(), motors.turn_left() etc.
    pass
